# Dikduk checker — test server
One cell. Runtime → Run all (or the ▶ on the cell). After about four minutes it prints two addresses: the lab page for you, and `/try` for someone else to test. The server lives while this notebook runs (Colab stops it after about 90 idle minutes or when the tab is closed for long). Nothing here touches the site.


In [ ]:
#@title ▶ Run this cell: it starts the dikduk checker and prints its address (about 4 minutes). Leave the tab open while people test.
import subprocess, time, re, os, sys, json
REPO = 'https://github.com/davidsitt810-ops/dikduk-space.git'
if not os.path.exists('/content/dk'):
    subprocess.run(['git', 'clone', '-q', REPO, '/content/dk'], check=True)
os.chdir('/content/dk')
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'fastapi', 'uvicorn', 'transformers==4.44.2', 'huggingface_hub', 'requests'], check=True)
subprocess.run('apt-get -qq install -y ffmpeg > /dev/null 2>&1', shell=True)
subprocess.run('wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /content/cloudflared', shell=True, check=True)
import torch
import secrets
KEY = secrets.token_hex(8)   # the address below is public: the booth's recordings and the throat drill need this key
env = dict(os.environ, DK_CACHE='/content/dkcache', DK_DEVICE='cuda' if torch.cuda.is_available() else 'cpu', DK_PUBLIC='1', DK_KEY=KEY)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none (CPU: a check takes 10-60 s)')
srv = subprocess.Popen([sys.executable, '-m', 'uvicorn', 'app:app', '--host', '127.0.0.1', '--port', '7861'], stdout=open('/content/server.log', 'w'), stderr=subprocess.STDOUT, env=env)
print('loading the models ...', end='', flush=True)
for i in range(240):
    time.sleep(5); log = open('/content/server.log').read()
    if 'Application startup complete' in log: break
    if srv.poll() is not None: print('\nthe server stopped; last lines:\n', log[-3000:]); raise SystemExit
    print('.', end='', flush=True)
print(' ready')
tun = subprocess.Popen(['/content/cloudflared', 'tunnel', '--url', 'http://127.0.0.1:7861', '--no-autoupdate'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
for line in tun.stdout:
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if m: url = m.group(0); break
print('\n' + '=' * 60)
print('LAB PAGE (you):        ', url)
print('FOR SOMEONE TO TRY IT: ', url + '/try')
print('KEY (keep it to yourself):', KEY, ' -> the lab page with the drill:', url + '/?key=' + KEY)
print('Recordings made here are gone when this notebook stops.')
print('=' * 60)
print('The address works as long as this cell keeps running. Stop it (■) when done; run it again for a new address.')
while True:
    time.sleep(60)
    if srv.poll() is not None: print('the server stopped:', open('/content/server.log').read()[-2000:]); break
